In [1]:
# =========================================
# TRAIN ALL MODEL-2 CLASSIFIERS IN ONE NOTEBOOK
# =========================================

import os
import json
import time
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import ImageFolder
import torchvision.transforms as T
from sklearn.metrics import confusion_matrix, classification_report
import timm
from tqdm import tqdm

# -----------------------------------------
# CONFIG
# -----------------------------------------
SEED = 42

# CHANGE THIS
MODEL2_ROOT = "/kaggle/input/datasets/rupeshbhardwaj420/hybrid-image-classifier-class-dataset/categories"
OUT_ROOT = Path("/kaggle/working/model2_outputs")
OUT_ROOT.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 8
LR = 3e-4
WEIGHT_DECAY = 1e-4
MODEL_NAME = "efficientnet_b0"
VAL_RATIO = 0.2
NUM_WORKERS = 2

# category folder names exactly as they exist in Kaggle dataset
TARGET_CATEGORIES = [
    "vehicle",
    "flower",
    "food",
    "monument",
    "weather",
    "flags",
    "shapes",
    "sports equipments",
    "fruits and vegetables"
]

# if you later normalize folder names, update the list accordingly

# -----------------------------------------
# REPRODUCIBILITY
# -----------------------------------------
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# -----------------------------------------
# TRANSFORMS
# -----------------------------------------
train_tf = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(10),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.02),
    T.ToTensor(),
    T.Normalize(mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)),
])

val_tf = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)),
])

training_summary = {}


def evaluate(model, val_loader, criterion, num_classes):
    model.eval()

    y_true = []
    y_pred = []
    top3_correct = 0
    total = 0
    running_val_loss = 0.0

    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            logits = model(x)
            loss = criterion(logits, y)

            pred1 = torch.argmax(logits, dim=1)
            top3 = torch.topk(logits, k=min(3, num_classes), dim=1).indices

            top3_correct += (top3 == y.unsqueeze(1)).any(dim=1).sum().item()
            total += y.size(0)
            running_val_loss += loss.item() * x.size(0)

            y_true.extend(y.cpu().tolist())
            y_pred.extend(pred1.cpu().tolist())

    top1 = (np.array(y_true) == np.array(y_pred)).mean()
    top3_acc = top3_correct / max(total, 1)
    avg_val_loss = running_val_loss / max(total, 1)
    cm = confusion_matrix(y_true, y_pred)

    return float(avg_val_loss), float(top1), float(top3_acc), cm, y_true, y_pred


for category_name in TARGET_CATEGORIES:
    print("\n" + "=" * 70)
    print(f"TRAINING CATEGORY MODEL: {category_name}")
    print("=" * 70)

    try:
        data_root = Path(MODEL2_ROOT) / category_name

        if not data_root.exists():
            raise FileNotFoundError(f"Category folder not found: {data_root}")

        # output folder for this category
        safe_category_name = category_name.replace(" ", "_").lower()
        out_dir = OUT_ROOT / f"{safe_category_name}_model"
        out_dir.mkdir(parents=True, exist_ok=True)

        # dataset
        full_train = ImageFolder(str(data_root), transform=train_tf)
        full_val = ImageFolder(str(data_root), transform=val_tf)

        n = len(full_train)
        idx = np.arange(n)
        np.random.shuffle(idx)

        val_n = int(n * VAL_RATIO)
        val_idx = idx[:val_n]
        train_idx = idx[val_n:]

        train_ds = Subset(full_train, train_idx)
        val_ds = Subset(full_val, val_idx)

        class_to_idx = full_train.class_to_idx
        idx_to_class = {v: k for k, v in class_to_idx.items()}
        num_classes = len(class_to_idx)

        print("Data root:", data_root)
        print("Total images:", n)
        print("Train images:", len(train_ds))
        print("Val images:", len(val_ds))
        print("Num classes:", num_classes)
        print("Class names:", list(class_to_idx.keys()))

        train_loader = DataLoader(
            train_ds,
            batch_size=BATCH_SIZE,
            shuffle=True,
            num_workers=NUM_WORKERS,
            pin_memory=True
        )

        val_loader = DataLoader(
            val_ds,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=NUM_WORKERS,
            pin_memory=True
        )

        # model
        model = timm.create_model(
            MODEL_NAME,
            pretrained=True,
            num_classes=num_classes
        )
        model = model.to(device)

        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=LR,
            weight_decay=WEIGHT_DECAY
        )

        best_top1 = -1.0
        best_path = out_dir / f"{safe_category_name}_model_best.pt"

        # train loop
        for epoch in range(1, EPOCHS + 1):
            model.train()
            running_loss = 0.0
            seen = 0
            t0 = time.time()

            for x, y in tqdm(train_loader, desc=f"{category_name} epoch {epoch}/{EPOCHS}"):
                x = x.to(device, non_blocking=True)
                y = y.to(device, non_blocking=True)

                optimizer.zero_grad(set_to_none=True)
                logits = model(x)
                loss = criterion(logits, y)
                loss.backward()
                optimizer.step()

                bs = x.size(0)
                running_loss += loss.item() * bs
                seen += bs

            train_loss = running_loss / max(seen, 1)
            val_loss, top1, top3, cm, y_true, y_pred = evaluate(
                model, val_loader, criterion, num_classes
            )
            dt = time.time() - t0

            print(
                f"{category_name} | Epoch {epoch}: "
                f"train_loss={train_loss:.4f} | "
                f"val_loss={val_loss:.4f} | "
                f"top1={top1:.4f} | "
                f"top3={top3:.4f} | "
                f"time={dt:.1f}s"
            )

            if top1 > best_top1:
                best_top1 = top1
                torch.save(model.state_dict(), best_path)
                print("✅ Saved best model:", best_path)

        # save last model
        last_path = out_dir / f"{safe_category_name}_model_last.pt"
        torch.save(model.state_dict(), last_path)

        # save metadata
        (out_dir / "class_to_idx.json").write_text(json.dumps(class_to_idx, indent=2))

        metrics = {
            "best_val_top1": best_top1,
            "model_name": MODEL_NAME,
            "image_size": IMAGE_SIZE,
            "batch_size": BATCH_SIZE,
            "epochs": EPOCHS,
            "num_classes": num_classes,
            "category_name": category_name
        }
        (out_dir / "metrics.json").write_text(json.dumps(metrics, indent=2))

        config_snapshot = {
            "seed": SEED,
            "data_root": str(data_root),
            "val_ratio": VAL_RATIO,
            "image_size": IMAGE_SIZE,
            "batch_size": BATCH_SIZE,
            "epochs": EPOCHS,
            "lr": LR,
            "weight_decay": WEIGHT_DECAY,
            "model_name": MODEL_NAME,
            "num_classes": num_classes,
            "category_name": category_name
        }
        (out_dir / f"config_{safe_category_name}.json").write_text(
            json.dumps(config_snapshot, indent=2)
        )

        report = classification_report(
            y_true,
            y_pred,
            target_names=[idx_to_class[i] for i in range(num_classes)],
            output_dict=True
        )
        (out_dir / "classification_report.json").write_text(
            json.dumps(report, indent=2)
        )

        training_summary[category_name] = {
            "status": "success",
            "best_val_top1": best_top1,
            "num_classes": num_classes,
            "output_dir": str(out_dir)
        }

        # save running summary after each category
        (OUT_ROOT / "training_summary.json").write_text(
            json.dumps(training_summary, indent=2)
        )

        print(f"✅ Finished training for {category_name}")
        print(f"✅ Output saved in {out_dir}")

        # free memory before next model
        del model
        torch.cuda.empty_cache()

    except Exception as e:
        print(f"❌ Failed category: {category_name}")
        print("Error:", str(e))

        training_summary[category_name] = {
            "status": "failed",
            "error": str(e)
        }

        (OUT_ROOT / "training_summary.json").write_text(
            json.dumps(training_summary, indent=2)
        )

        torch.cuda.empty_cache()
        continue

print("\n" + "=" * 70)
print("ALL TRAINING DONE")
print("=" * 70)
print(json.dumps(training_summary, indent=2))

Device: cuda

TRAINING CATEGORY MODEL: vehicle
Data root: /kaggle/input/datasets/rupeshbhardwaj420/hybrid-image-classifier-class-dataset/categories/vehicle
Total images: 1000
Train images: 800
Val images: 200
Num classes: 4
Class names: ['bus', 'car', 'motorcycle', 'truck']


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

vehicle epoch 1/8: 100%|██████████| 25/25 [00:06<00:00,  3.58it/s]


vehicle | Epoch 1: train_loss=1.4110 | val_loss=0.8165 | top1=0.8100 | top3=0.9850 | time=8.1s
✅ Saved best model: /kaggle/working/model2_outputs/vehicle_model/vehicle_model_best.pt


vehicle epoch 2/8: 100%|██████████| 25/25 [00:04<00:00,  5.02it/s]


...

fruits and vegetables | Epoch 6: train_loss=0.0805 | val_loss=0.5913 | top1=0.8405 | top3=0.9490 | time=92.7s


fruits and vegetables epoch 7/8:   4%|▍         | 4/96 [00:04<01:15,  1.22it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
fruits and vegetables epoch 7/8:  17%|█▋        | 16/96 [00:13<00:51,  1.54it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
fruits and vegetables epoch 7/8: 100%|██████████| 96/96 [01:15<00:00,  1.28it/s]
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


fruits and vegetables | Epoch 7: train_loss=0.0700 | val_loss=0.5715 | top1=0.8523 | top3=0.9451 | time=92.7s


fruits and vegetables epoch 8/8:   2%|▏         | 2/96 [00:02<01:20,  1.16it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
fruits and vegetables epoch 8/8:   9%|▉         | 9/96 [00:07<00:55,  1.58it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
fruits and vegetables epoch 8/8: 100%|██████████| 96/96 [01:16<00:00,  1.25it/s]
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


fruits and vegetables | Epoch 8: train_loss=0.0516 | val_loss=0.6064 | top1=0.8405 | top3=0.9490 | time=95.4s
✅ Finished training for fruits and vegetables
✅ Output saved in /kaggle/working/model2_outputs/fruits_and_vegetables_model

ALL TRAINING DONE
{
  "vehicle": {
    "status": "success",
    "best_val_top1": 0.84,
    "num_classes": 4,
    "output_dir": "/kaggle/working/model2_outputs/vehicle_model"
  },
  "flower": {
    "status": "success",
    "best_val_top1": 0.828,
    "num_classes": 10,
    "output_dir": "/kaggle/working/model2_outputs/flower_model"
  },
  "food": {
    "status": "success",
    "best_val_top1": 0.811,
    "num_classes": 20,
    "output_dir": "/kaggle/working/model2_outputs/food_model"
  },
  "monument": {
    "status": "success",
    "best_val_top1": 0.9859649122807017,
    "num_classes": 12,
    "output_dir": "/kaggle/working/model2_outputs/monument_model"
  },
  "weather": {
    "status": "success",
    "best_val_top1": 0.9585492227979274,
    "num_classes